# 12 Error Analysis

Το notebook αναλύει συστηματικά τα σφάλματα των pipelines, με έμφαση στις περιπτώσεις όπου η ανάκτηση ή η τελική απάντηση δεν συμφωνεί με το αναμενόμενο αποτέλεσμα του FinanceBench.


In [ ]:
import json
import re
from pathlib import Path

import pandas as pd

from scripts.evaluation_metrics import (
    evaluate_answer_record,
    is_insufficient_evidence,
    lexical_f1,
)


## 1. Configuration

In [ ]:
# ── Επιλογή run για ανάλυση ──────────────────────────────────────────────────
PRIMARY_RUN = "dense"
ALL_RUNS    = ["dense", "hybrid", "hybrid_reranked"]

# ── Threshold για το finance-aware score ώστε να θεωρείται σωστή η απάντηση ───
ANSWER_CORRECT_SCORE_THRESHOLD = 0.3

# ── Top-K που χρησιμοποιήθηκε στο retrieval ─────────────────────────────────
TOP_K = 5

# ── Paths (ίδιο σύστημα με τα υπόλοιπα notebooks) ───────────────────────────
CURRENT_DIR = Path.cwd()
BASE_DIR    = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

DATA_DIR      = BASE_DIR / "data"
INTERIM_DIR   = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
QA_DIR        = PROCESSED_DIR / "qa_results"
RETRIEVAL_DIR = PROCESSED_DIR / "retrieval_results"
EVAL_DIR      = PROCESSED_DIR / "evaluation"
ERROR_DIR     = PROCESSED_DIR / "error_analysis"

ERROR_DIR.mkdir(parents=True, exist_ok=True)

WORKING_DATASET_PATH = (
    INTERIM_DIR / "financebench_open_source_working.parquet"
    if (INTERIM_DIR / "financebench_open_source_working.parquet").exists()
    else INTERIM_DIR / "financebench_open_source_working.csv"
)

print("BASE_DIR   :", BASE_DIR)
print("ERROR_DIR  :", ERROR_DIR)
print("PRIMARY_RUN:", PRIMARY_RUN)

## 2. Φόρτωση δεδομένων

In [ ]:
# ── Gold dataset ─────────────────────────────────────────────────────────────
if WORKING_DATASET_PATH.suffix == ".parquet":
    try:
        working_df = pd.read_parquet(WORKING_DATASET_PATH)
    except ImportError:
        working_df = pd.read_csv(WORKING_DATASET_PATH.with_suffix(".csv"))
else:
    working_df = pd.read_csv(WORKING_DATASET_PATH)

print(f"working_df   : {working_df.shape}")

# ── QA results ───────────────────────────────────────────────────────────────
qa_dfs = {}
for run in ALL_RUNS:
    p = QA_DIR / f"rag_qa_results_{run}.csv"
    if p.exists():
        qa_dfs[run] = pd.read_csv(p)
        print(f"QA  {run:20s}: {qa_dfs[run].shape}")
    else:
        print(f"QA  {run:20s}: δεν βρέθηκε — {p}")

# ── Retrieval results ─────────────────────────────────────────────────────────
retrieval_dfs = {}
for run in ALL_RUNS:
    parquet_path = RETRIEVAL_DIR / f"retrieval_results_{run}.parquet"
    csv_path = RETRIEVAL_DIR / f"retrieval_results_{run}.csv"
    if parquet_path.exists():
        try:
            retrieval_dfs[run] = pd.read_parquet(parquet_path)
        except ImportError:
            retrieval_dfs[run] = pd.read_csv(csv_path)
    elif csv_path.exists():
        retrieval_dfs[run] = pd.read_csv(csv_path)
    else:
        print(f"RET {run:20s}: δεν βρέθηκε")
        continue
    print(f"RET {run:20s}: {retrieval_dfs[run].shape}")


## 3. Βοηθητικές συναρτήσεις

In [ ]:
# Shared finance-aware metrics are imported from scripts.evaluation_metrics.

def extract_company_from_doc(doc_name: str) -> str:
    """Extract company key from a FinanceBench document id."""
    if pd.isna(doc_name):
        return ""
    parts = str(doc_name).split("_")
    return parts[0].upper() if parts else ""


print("Shared evaluation helpers loaded.")


## 4. Ταξινομητής αποτυχιών

Η κεντρική συνάρτηση `classify_failure()` εφαρμόζει ένα **decision tree** ταξινόμησης:  
ελέγχει διαδοχικά (α) retrieval miss, (β) άρνηση απάντησης, (γ) λάθος έτος/doc, (δ) generation failure.

In [ ]:
# Κατηγορίες αποτυχίας
FAILURE_CATEGORIES = [
    "SUCCESS",
    "RETRIEVAL_MISS",          # Το σωστό doc απουσιάζει εντελώς από τα top-K chunks
    "WRONG_YEAR_DOC",          # Σωστή εταιρεία, λάθος έτος ή τύπος εγγράφου
    "GOOD_RETRIEVAL_BAD_ANSWER",  # Σωστό doc ανακτάται, λάθος απάντηση από LLM
    "INSUFFICIENT_EVIDENCE",   # LLM αρνήθηκε να απαντήσει
]


def classify_failure(
    expected_doc: str,
    retrieved_docs: list,       # λίστα με τα doc_ids που ανακτήθηκαν (top-K)
    gold_answer: str,
    generated_answer: str,
    question: str = "",
    score_threshold: float = ANSWER_CORRECT_SCORE_THRESHOLD,
) -> str:
    """
    Ταξινομεί κάθε ερώτηση σε μία κατηγορία αποτυχίας.

    Decision tree:
    1. Αν finance-aware answer score >= threshold → SUCCESS
    2. Αν expected_doc δεν υπάρχει στα retrieved_docs → RETRIEVAL_MISS
    3. Αν LLM επέστρεψε 'Insufficient evidence' → INSUFFICIENT_EVIDENCE
    4. Αν expected_doc υπάρχει στα retrieved_docs (σωστή εταιρεία, λάθος f1)
       → GOOD_RETRIEVAL_BAD_ANSWER
    5. Αν η εταιρεία ταιριάζει αλλά το exact doc όχι → WRONG_YEAR_DOC
    6. Fallback → RETRIEVAL_MISS
    """
    answer_metrics = evaluate_answer_record(
        prediction=generated_answer,
        reference=gold_answer,
        question=question,
    )
    answer_score = answer_metrics["finance_aware_score"]

    # Βήμα 1: Σωστή απάντηση
    if answer_score >= score_threshold:
        return "SUCCESS"

    expected_doc = str(expected_doc) if not pd.isna(expected_doc) else ""
    retrieved_set = {str(d) for d in retrieved_docs if not pd.isna(d)}

    # Βήμα 2: Retrieval miss — το σωστό doc απουσιάζει εντελώς
    doc_retrieved = expected_doc in retrieved_set

    # Βήμα 3: LLM αρνήθηκε (ακόμα και αν το doc ανακτήθηκε)
    if is_insufficient_evidence(generated_answer):
        if not doc_retrieved:
            return "RETRIEVAL_MISS"
        else:
            return "INSUFFICIENT_EVIDENCE"

    # Βήμα 4: Σωστό doc ανακτάται αλλά το LLM απαντά λάθος
    if doc_retrieved:
        return "GOOD_RETRIEVAL_BAD_ANSWER"

    # Βήμα 5: Σωστή εταιρεία, λάθος έτος/doc
    expected_company = extract_company_from_doc(expected_doc)
    retrieved_companies = {extract_company_from_doc(d) for d in retrieved_set}
    if expected_company and expected_company in retrieved_companies:
        return "WRONG_YEAR_DOC"

    # Βήμα 6: Fallback
    return "RETRIEVAL_MISS"


print("Ταξινομητής ορίστηκε.")

## 5. Εφαρμογή ταξινόμησης στο primary run

In [ ]:
assert PRIMARY_RUN in qa_dfs, f"{PRIMARY_RUN} δεν βρέθηκε στα QA αποτελέσματα."
assert PRIMARY_RUN in retrieval_dfs, f"{PRIMARY_RUN} δεν βρέθηκε στα retrieval αποτελέσματα."

qa_df  = qa_dfs[PRIMARY_RUN].copy()
ret_df = retrieval_dfs[PRIMARY_RUN].copy()

# ── Δημιουργία per-query λίστας retrieved doc_ids ───────────────────────────
# Διατηρούνται τα TOP_K ανακτηθέντα docs ανά ερώτηση
retrieved_docs_per_query = (
    ret_df[ret_df["retrieved_rank"] <= TOP_K]
    .groupby("financebench_id")["retrieved_doc_id"]
    .apply(list)
    .reset_index()
    .rename(columns={"retrieved_doc_id": "retrieved_doc_list"})
)

# ── Merge QA + retrieval ──────────────────────────────────────────────────────
analysis_df = qa_df.merge(retrieved_docs_per_query, on="financebench_id", how="left")
analysis_df["retrieved_doc_list"] = analysis_df["retrieved_doc_list"].apply(
    lambda x: x if isinstance(x, list) else []
)

# ── Εφαρμογή ταξινόμησης ─────────────────────────────────────────────────────
analysis_df["failure_category"] = analysis_df.apply(
    lambda row: classify_failure(
        expected_doc      = row.get("expected_doc_name", ""),
        retrieved_docs    = row["retrieved_doc_list"],
        gold_answer       = row.get("expected_answer", ""),
        generated_answer  = row.get("generated_answer", ""),
        question          = row.get("question", ""),
    ),
    axis=1,
)

# ── Shared finance-aware metrics per question ─────────────────────────────────
answer_metric_df = analysis_df.apply(
    lambda row: pd.Series(evaluate_answer_record(
        prediction=row.get("generated_answer", ""),
        reference=row.get("expected_answer", ""),
        question=row.get("question", ""),
        context=row.get("context_text", ""),
    )),
    axis=1,
)
analysis_df = pd.concat([analysis_df, answer_metric_df], axis=1)

print(f"analysis_df shape: {analysis_df.shape}")
print("\nΚατανομή κατηγοριών:")
print(analysis_df["failure_category"].value_counts())

## 6. Συνοπτική αναφορά

In [ ]:
category_counts = analysis_df["failure_category"].value_counts()
total = len(analysis_df)

# ── Δημιουργία summary με το σωστό column name από την αρχή ──────────────────
summary_df = pd.DataFrame({
    "failure_category": category_counts.index,
    "count"           : category_counts.values,
    "percentage"      : (category_counts.values / total * 100).round(1),
})

# ── Μέσο F1 ανά κατηγορία (απευθείας merge, χωρίς conditional) ───────────────
mean_score_df = (
    analysis_df
    .groupby("failure_category", as_index=False)["finance_aware_score"]
    .mean()
    .rename(columns={"finance_aware_score": "mean_finance_aware_score"})
)
mean_score_df["mean_finance_aware_score"] = mean_score_df["mean_finance_aware_score"].round(3)

summary_df = summary_df.merge(mean_score_df, on="failure_category", how="left")

print(f"\n{'='*60}")
print(f"  Error Analysis Summary — {PRIMARY_RUN} (N={total})")
print(f"{'='*60}")
print(summary_df.to_string(index=False))
print(f"{'='*60}\n")

summary_df

## 7. Ανάλυση ανά κατηγορία ερώτησης (question_type)

Εξετάζεται αν κάποιος τύπος ερώτησης (π.χ. `extractive`, `arithmetic`) παρουσιάζει  
συστηματικά υψηλότερα ποσοστά αποτυχίας — χρήσιμο για το ablation study.

In [ ]:
# Merge με το working_df για την ανάκτηση των question_type, question_reasoning, gics_sector
meta_cols = ["financebench_id", "question_type", "question_reasoning", "gics_sector", "doc_type"]
available_meta = [c for c in meta_cols if c in working_df.columns]

rich_df = analysis_df.merge(
    working_df[available_meta].drop_duplicates(subset=["financebench_id"]),
    on="financebench_id",
    how="left",
)

# ── Αποτυχίες ανά question_type ──────────────────────────────────────────────
if "question_type" in rich_df.columns:
    print("\n── Αποτυχίες ανά question_type ──")
    qt_pivot = pd.crosstab(
        rich_df["question_type"],
        rich_df["failure_category"],
        margins=True,
        margins_name="TOTAL",
    )
    print(qt_pivot)

# ── Αποτυχίες ανά doc_type (10K vs 10Q) ─────────────────────────────────────
if "doc_type" in rich_df.columns:
    print("\n── Αποτυχίες ανά doc_type ──")
    dt_pivot = pd.crosstab(
        rich_df["doc_type"],
        rich_df["failure_category"],
        margins=True,
        margins_name="TOTAL",
    )
    print(dt_pivot)

## 8. Qualitative examples ανά κατηγορία

Για κάθε κατηγορία αποτυχίας, εμφανίζονται **3 αντιπροσωπευτικά παραδείγματα**  
για ποιοτική ανάλυση (θα ενσωματωθούν στο κεφάλαιο Ευρημάτων).

In [ ]:
display_cols = [
    "financebench_id",
    "question",
    "expected_answer",
    "generated_answer",
    "expected_doc_name",
    "retrieved_doc_list",
    "lexical_f1",
    "failure_category",
]
display_cols = [c for c in display_cols if c in rich_df.columns]

for category in FAILURE_CATEGORIES:
    subset = rich_df[rich_df["failure_category"] == category]
    if len(subset) == 0:
        continue

    print(f"\n{'─'*70}")
    print(f"  {category}  (N={len(subset)})")
    print(f"{'─'*70}")

    examples = subset.sample(min(3, len(subset)), random_state=42)
    for _, row in examples.iterrows():
        print(f"\n  ID      : {row.get('financebench_id', '')}")
        q = str(row.get('question', ''))[:120]
        print(f"  Question: {q}..." if len(str(row.get('question',''))) > 120 else f"  Question: {q}")
        print(f"  Gold    : {str(row.get('expected_answer', ''))[:100]}")
        print(f"  Pred    : {str(row.get('generated_answer', ''))[:100]}")
        print(f"  Score   : {row.get('finance_aware_score', 0):.3f}")
        print(f"  Exp.doc : {row.get('expected_doc_name', '')}")
        retrieved = row.get('retrieved_doc_list', [])
        if isinstance(retrieved, list):
            print(f"  Ret.docs: {retrieved[:5]}")

## 9. Σύγκριση αποτυχιών μεταξύ όλων των runs

Εξετάζεται πώς εξελίσσεται η κατανομή κατηγοριών καθώς βελτιώνεται το pipeline:  
`dense` → `hybrid` → `hybrid_reranked`.

In [ ]:
cross_run_rows = []

for run in ALL_RUNS:
    if run not in qa_dfs or run not in retrieval_dfs:
        print(f"Παράλειψη {run}: λείπουν δεδομένα")
        continue

    _qa  = qa_dfs[run].copy()
    _ret = retrieval_dfs[run].copy()

    _ret_docs = (
        _ret[_ret["retrieved_rank"] <= TOP_K]
        .groupby("financebench_id")["retrieved_doc_id"]
        .apply(list)
        .reset_index()
        .rename(columns={"retrieved_doc_id": "retrieved_doc_list"})
    )

    _df = _qa.merge(_ret_docs, on="financebench_id", how="left")
    _df["retrieved_doc_list"] = _df["retrieved_doc_list"].apply(
        lambda x: x if isinstance(x, list) else []
    )

    _df["failure_category"] = _df.apply(
        lambda row: classify_failure(
            expected_doc     = row.get("expected_doc_name", ""),
            retrieved_docs   = row["retrieved_doc_list"],
            gold_answer      = row.get("expected_answer", ""),
            generated_answer = row.get("generated_answer", ""),
            question         = row.get("question", ""),
        ),
        axis=1,
    )

    for cat, cnt in _df["failure_category"].value_counts().items():
        cross_run_rows.append({
            "run"             : run,
            "failure_category": cat,
            "count"           : int(cnt),
            "percentage"      : round(cnt / len(_df) * 100, 1),
        })

cross_run_df = pd.DataFrame(cross_run_rows)

# Pivot για ευκολότερη ανάγνωση
pivot = cross_run_df.pivot_table(
    index="failure_category",
    columns="run",
    values="percentage",
    aggfunc="first",
).fillna(0)

# Επαναδιατύπωση στη σειρά dense → hybrid → hybrid_reranked
ordered_runs = [r for r in ALL_RUNS if r in pivot.columns]
pivot = pivot[ordered_runs]

print("\nΠοσοστά (%) ανά κατηγορία και run:")
print(pivot.to_string())

## 10. Αποθήκευση αποτελεσμάτων

In [ ]:
# ── Full analysis table ───────────────────────────────────────────────────────
save_cols = [
    "financebench_id", "question", "expected_answer", "generated_answer",
    "expected_doc_name", "top_doc_id", "finance_aware_score", "lexical_f1", "failure_category",
]
save_cols = [c for c in save_cols if c in rich_df.columns]

out_full = ERROR_DIR / f"error_analysis_{PRIMARY_RUN}.csv"
rich_df[save_cols].to_csv(out_full, index=False)
print(f"Αποθηκεύτηκε: {out_full}")

# ── Summary ────────────────────────────────────────────────────────────────────
out_summary = ERROR_DIR / f"error_summary_{PRIMARY_RUN}.csv"
summary_df.to_csv(out_summary, index=False)
print(f"Αποθηκεύτηκε: {out_summary}")

# ── Cross-run comparison ───────────────────────────────────────────────────────
out_cross = ERROR_DIR / "error_cross_run_comparison.csv"
cross_run_df.to_csv(out_cross, index=False)
print(f"Αποθηκεύτηκε: {out_cross}")

# ── JSON stats (για εύκολη αναφορά στη διπλωματική) ──────────────────────────
stats = {
    "run"            : PRIMARY_RUN,
    "total_queries"  : int(total),
    "score_threshold": ANSWER_CORRECT_SCORE_THRESHOLD,
    "top_k"          : TOP_K,
    "category_counts": {
        row["failure_category"]: {
            "count"     : int(row["count"]),
            "percentage": float(row["percentage"]),
        }
        for _, row in summary_df.iterrows()
    },
}

out_json = ERROR_DIR / f"error_stats_{PRIMARY_RUN}.json"
with open(out_json, "w", encoding="utf-8") as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)
print(f"Αποθηκεύτηκε: {out_json}")

print("\n✓ Error analysis ολοκληρώθηκε.")